# 手动标题解析辅助工具

## 目的

逐行浏览 `music.csv`（按播放量降序），帮助用户快速建立**关键词→标准歌名+原唱者**的映射数据库。

映射库格式（`data/song_keywords.csv`）：
```csv
keyword,song_name,original_creator
六兆年,六兆年と一夜物語,kemu
Children Record,六兆年と一夜物語,kemu
哨戒班,アスノヨゾラ哨戒班,Orangestar
```

## 为什么这样做比纯正则更好

1. **知名歌曲反复出现**：热门歌（六兆年、哨戒班、チルドレンレコード）有几十个版本，一次标记永久生效
2. **人眼判断 P主 准确**：看到 `鳥の詩` 知道是 Lia，看到 `Henceforth` 知道是 Orangestar
3. **关键词比全歌名更灵活**：'哨戒班' 匹配 "アスノヨゾラ哨戒班" 和 "明日的夜空哨戒班"
4. **为 LLM 提供训练样本**：标记过的数据可以作为 LLM fine-tuning 的标注集

## 工作流

```
程序启动 → 加载 music.csv + 已有 song_keywords.csv
           ↓
逐行展示（播放量从高到低）:
  ┌────────────────────────────────────────┐
  │ #42  BV1ax411w7gM  播放: 1,927,008     │
  │ title: 【IA】六兆年と一夜物語【kemu】    │
  │ tags:  KE-SANβ,VOCALOID3,kemu,IA,...   │
  │ auther: Lynn_Campbell_                  │
  │                                        │
  │ 已有映射命中: kemu → 六兆年と一夜物語    │ ← 如果匹配到已有keyword
  │ 自动建议: 六兆年と一夜物語 (by kemu)     │ ← 推荐摘要
  ├────────────────────────────────────────┤
  │ [s]跳过  [n]新建歌曲  [a]加到已有歌曲   │
  │ [e]编辑当前行  [q]保存并退出            │
  └────────────────────────────────────────┘
```

## 核心功能

### 1. 自动建议

进入每行时，自动检查标题是否命中已有 keyword 映射：

```python
# 如果标题含 "六兆年"，且 song_keywords.csv 有 "六兆年" → "六兆年と一夜物語":"kemu"
# 显示: ⚡ 已匹配: keyword='六兆年' → 六兆年と一夜物語 (kemu)
# 如果匹配多个 → 列出所有匹配，用户选择
```

### 2. 新建歌曲 [n]

```
输入关键词 (逗号分隔): 哨戒班, 明日的夜空
输入标准歌名: アスノヨゾラ哨戒班
输入原唱者 (P主): Orangestar
→ 写入 song_keywords.csv: 3行
  哨戒班,アスノヨゾラ哨戒班,Orangestar
  明日的夜空,アスノヨゾラ哨戒班,Orangestar
```

### 3. 加到已有歌曲 [a]

```
显示已有歌曲列表 (按歌名字母序):
  1. アスノヨゾラ哨戒班 (Orangestar)
  2. 六兆年と一夜物語 (kemu)
  3. ...
选择编号: 2
输入新关键词: Children Record
→ 追加: Children Record,六兆年と一夜物語,kemu
```

### 4. 跳过 [s]
已正确识别的歌曲直接跳过。统计显示"已标注: N, 跳过: M"。

### 5. 保存并退出 [q]
自动保存 `song_keywords.csv` + 进度记录（当前行号）。

## 具体实现细节

### 数据加载

```python
# 加载 music.csv，按播放量降序
import pandas as pd
df = pd.read_csv('data/labeled_samples/music.csv', on_bad_lines='skip', engine='python')
df = df.sort_values('play_count', ascending=False)

# 加载已有映射（如果存在）
if os.path.exists('data/song_keywords.csv'):
    mapping = pd.read_csv('data/song_keywords.csv')
else:
    mapping = pd.DataFrame(columns=['keyword','song_name','original_creator'])

# 加载进度
if os.path.exists('data/.parser_progress'):
    start_row = int(open('data/.parser_progress').read())
else:
    start_row = 0
```

### 行展示格式

关键细节：
- 标题中的 `【IA】` 部分用颜色高亮（如果终端支持）或加 `>>> 【IA】<<<` 标记
- tags 按逗号拆分后限制显示长度（超过 80 字符截断）
- 如果有自动匹配，用星号或箭头标注

```python
def display_row(row, index, total, matches):
    print(f'\n{"="*70}')
    print(f'[{index+1}/{total}] BV: {row["bvid"]}  |  播放: {row["play_count"]:,}')
    print(f'标题: {row["title"]}')
    print(f'标签: {row["tags"][:80]}')
    print(f'UP主: {row["author"]}')
    if matches:
        print(f'\n⚡ 已有匹配:')
        for m in matches:
            print(f'   keyword="{m["keyword"]}" → {m["song_name"]} by {m["original_creator"]}')
```

### 自动匹配逻辑

```python
def find_matches(title, mapping):
    '''检查标题是否含已有的keyword'''
    matches = []
    for _, row in mapping.iterrows():
        if row['keyword'].upper() in title.upper():
            matches.append(row.to_dict())
    return matches
```

注意事项：
- 大小写不敏感匹配
- 中文 keyword 也是子串匹配（"哨戒班" 匹配 "アスノヨゾラ哨戒班"）
- 如果 keyword 太短（<3字）可能误匹配，建议 program 拒绝过短的 keyword

### 新建歌曲逻辑

```python
def new_song(title, mapping):
    print('输入关键词（逗号分隔，可从标题复制部分):')
    keywords = input('> ').split(',')
    keywords = [k.strip() for k in keywords if len(k.strip()) >= 2]
    
    song_name = input('标准歌名: ').strip()
    creator = input('原唱者(P主): ').strip()
    
    for kw in keywords:
        mapping.loc[len(mapping)] = [kw, song_name, creator]
    
    print(f'✓ 已添加 {len(keywords)} 个关键词 → {song_name}')
    return mapping
```

注意事项：
- keyword 最小长度 2 字符（避免"IA"这种太短的误匹配）
- 如果 keyword 已存在，提示"已存在，是否覆盖？"
- 建议用户从标题中复制最独特的片段作为 keyword

### 进度保存

```python
def save_progress(mapping, current_index):
    # 保存映射
    mapping.to_csv('data/song_keywords.csv', index=False, encoding='utf-8-sig')
    # 保存进度
    with open('data/.parser_progress', 'w') as f:
        f.write(str(current_index))
    print(f'✓ 已保存: {len(mapping)} 条映射, 进度 {current_index}')
```

关键：每次退出都保存进度，下次启动从断点继续。

### 快速工作建议

1. **先处理播放量高的**：前 200 首热门歌几乎覆盖了所有知名歌曲，后面的 80% 都是冷门
2. **批量跳过**：遇到已经正确识别的行（有自动匹配命中），快速按 `s` 跳过
3. **关键词选独特性强的**：'哨戒班' 比 'アスノ' 更独特（'アスノ' 可能匹配其他歌）
4. **用 vim/less 辅助**：如果终端显示混乱，可以加一个 `export to CSV` 功能，在 Excel 里批量编辑

## 复杂度评估

| 组件 | 代码量 | 难度 |
|------|:--:|:--:|
| 数据加载 + 排序 | ~15行 | 简单 |
| 逐行展示 | ~20行 | 简单 |
| 自动匹配 | ~10行 | 简单 |
| 新建歌曲 [n] | ~15行 | 简单 |
| 加到已有 [a] | ~15行 | 简单 |
| 主循环 + 进度保存 | ~25行 | 简单 |
| 总计 | **~100行** | 低 |

纯 Python 脚本，无 GUI，终端交互。

额外可选功能（增加复杂度但提升体验）：
- 未保存退出确认（+5行）
- 撤销上一个操作（+15行）
- 查看已有映射统计（+10行）
- 导出标注进度报告（+15行）